# Data Loading Functions Example

This notebook demonstrates all the data loading functions available in the ethopy_analysis package. Each function is explained with its purpose, parameters, and example usage.

In [ ]:
# Import all necessary modules
from ethopy_analysis.data.loaders import (
    get_sessions,
    get_trials,
    get_trial_states,
    get_trial_experiment,
    get_trial_behavior,
    get_trial_stimulus,
    get_trial_licks,
    get_trial_proximities,
    get_session_classes,
    get_session_duration,
    get_session_task,
    get_session_version
)
from ethopy_analysis.data.analysis import (
    get_performance,
    trials_per_session,
    session_summary
)
from ethopy_analysis.data.utils import get_setup
from ethopy_analysis.db.schemas import get_schema, get_all_schemas
from ethopy_analysis.config.settings import load_config

# Apply styling
from ethopy_analysis.config.styles import Style
Style().apply()

## 1. Configuration and Setup Functions

These functions handle configuration loading and database setup.

In [ ]:
# load_config() - Load configuration from file or use defaults
# Parameters: config_path (str/Path, optional)
# Returns: Configuration dictionary
config = load_config(display_path=True)
print("Configuration loaded:")
print(f"Database host: {config.get('database', {}).get('host', 'Not configured')}")
print(f"Available schemas: {list(config.get('database', {}).get('schemas', {}).keys())}")

In [ ]:
# get_setup() - Retrieve animal_id and session for a given setup
# Parameters: setup (str)
# Returns: Tuple[int, int] - (animal_id, session)
animal_id, session = get_setup("ef-rp13")
print(f"Setup 'ef-rp13' current animal_id: {animal_id}, session: {session}")

## 2. Database Schema Functions

These functions provide access to DataJoint database schemas.

In [ ]:
# get_schema() - Get a specific schema by name
# Parameters: schema_name (str: 'experiment', 'behavior', 'stimulus'), config (Dict, optional)
# Returns: DataJoint virtual module for the schema
experiment = get_schema('experiment')
behavior = get_schema('behavior')
stimulus = get_schema('stimulus')

print("Individual schemas loaded:")
print(f"Experiment schema: {type(experiment)}")
print(f"Behavior schema: {type(behavior)}")
print(f"Stimulus schema: {type(stimulus)}")

In [ ]:
# get_all_schemas() - Get all three schemas at once
# Parameters: config (Dict, optional)
# Returns: Dict with keys 'experiment', 'behavior', 'stimulus'
schemas = get_all_schemas()
print("All schemas loaded:")
for schema_name, schema_module in schemas.items():
    print(f"{schema_name}: {type(schema_module)}")

## 3. Session-Level Data Loading

These functions load session-level information and metadata.

In [ ]:
# get_sessions() - Get sessions for an animal within a date range
# Parameters: animal_id (int), from_date (str), to_date (str), format (str), min_trials (int)
# Returns: Session DataFrame or DataJoint expression
sessions = get_sessions(animal_id, min_trials=20)
print(f"Sessions for animal {animal_id} (min 20 trials):")
print(sessions.head())
print(f"\nTotal sessions: {len(sessions)}")
print(f"Session range: {sessions['session'].min()} to {sessions['session'].max()}")

In [ ]:
# get_sessions() with date filtering
# Filter sessions by date range
sessions_filtered = get_sessions(animal_id, from_date="2025-01-01", to_date="2025-12-31")
print(f"Sessions for animal {animal_id} in 2025:")
print(sessions_filtered.head())
print(f"Total sessions in 2025: {len(sessions_filtered)}")

In [ ]:
# get_session_classes() - Retrieve session information and experimental classes
# Parameters: animal_id (int), session (int)
# Returns: DataFrame with session info and class combinations
session_classes = get_session_classes(animal_id, session)
print(f"Session classes for animal {animal_id}, session {session}:")
print(session_classes)

In [ ]:
# get_session_duration() - Calculate session duration
# Parameters: animal_id (int), session (int)
# Returns: Formatted duration string or None
duration = get_session_duration(animal_id, session)
print(f"Session {session} duration: {duration}")

In [ ]:
# get_session_task() - Retrieve task configuration file
# Parameters: animal_id (int), session (int), save_file (bool)
# Returns: the task filename (str)
filename = get_session_task(animal_id, session, save_file=False)
print(f"Task file: {filename}")

# get_session_version() - Code version records for the session.
# One row per project directory; empty if the session has no records.
versions = get_session_version(animal_id, session)
if versions.empty:
    print("Code version: not recorded for this session")
else:
    display(versions[["project_path", "source_type", "version", "is_dirty"]])

## 4. Trial-Level Data Loading

These functions load detailed trial-level data for analysis.

In [ ]:
# get_trials() - Retrieve trial data for a specific session
# Parameters: animal_id (int), session (int), format (str), remove_abort (bool)
# Returns: Trial DataFrame or DataJoint expression
trials = get_trials(animal_id, session)
print(f"Trials for animal {animal_id}, session {session}:")
print(trials.head())
print(f"\nTotal trials: {len(trials)}")
print(f"Trial columns: {list(trials.columns)}")

In [ ]:
# get_trials() with abort removal
# Remove aborted trials from the dataset
trials_no_abort = get_trials(animal_id, session, remove_abort=True)
print(f"Trials without aborts: {len(trials_no_abort)} (vs {len(trials)} with aborts)")
print("Trial outcomes:")
print(trials_no_abort)

In [ ]:
# get_trial_states() - Retrieve trial state onset data
# Parameters: animal_id (int), session (int), format (str)
# Returns: Trial states DataFrame with state onset times
trial_states = get_trial_states(animal_id, session)
print(f"Trial states for animal {animal_id}, session {session}:")
print(trial_states.head())
print(f"\nUnique states: {trial_states['state'].unique()}")
print(f"Total state events: {len(trial_states)}")

In [ ]:
# get_trial_experiment() - Retrieve trial experiment condition data
# Parameters: animal_id (int), session (int), format (str)
# Returns: Trial experiment conditions DataFrame
trial_experiment = get_trial_experiment(animal_id, session)
print(f"Trial experiment data for animal {animal_id}, session {session}:")
print(trial_experiment.head())
print(f"\nExperiment columns: {list(trial_experiment.columns)}")

In [ ]:
# get_trial_behavior() - Retrieve trial behavior condition data
# Parameters: animal_id (int), session (int), format (str)
# Returns: Trial behavior conditions DataFrame
trial_behavior = get_trial_behavior(animal_id, session)
print(f"Trial behavior data for animal {animal_id}, session {session}:")
print(trial_behavior.head())
print(f"\nBehavior columns: {list(trial_behavior.columns)}")

In [ ]:
# get_trial_stimulus() - Retrieve trial stimulus condition data
# Parameters: animal_id (int), session (int), stim_class (str), format (str)
# Returns: Trial stimulus conditions DataFrame
trial_stimulus = get_trial_stimulus(animal_id, session)
print(f"Trial stimulus data for animal {animal_id}, session {session}:")
print(trial_stimulus.head())
print(f"\nStimulus columns: {list(trial_stimulus.columns)}")

## 5. Behavioral Event Data Loading

These functions load specific behavioral events like licks and proximity sensor data.

In [ ]:
# get_trial_licks() - Retrieve all licks of a session
# Parameters: animal_id (int), session (int), format (str)
# Returns: Lick events DataFrame
trial_licks = get_trial_licks(animal_id, session)
print(f"Trial licks for animal {animal_id}, session {session}:")
print(trial_licks.head())
print(f"\nTotal lick events: {len(trial_licks)}")
print(f"Lick columns: {list(trial_licks.columns)}")
print(f"Licks per port: {trial_licks['port'].value_counts()}")

In [ ]:
# get_trial_proximities() - Retrieve proximity sensor data
# Parameters: animal_id (int), session (int), ports (List), format (str)
# Returns: Proximity data DataFrame
trial_proximities = get_trial_proximities(animal_id, session)
print(f"Trial proximities for animal {animal_id}, session {session}:")
print(trial_proximities.head())
print(f"\nTotal proximity events: {len(trial_proximities)}")
print(f"Proximity columns: {list(trial_proximities.columns)}")
print(f"Proximity per port: {trial_proximities['port'].value_counts()}")

In [ ]:
# get_trial_proximities() with port filtering
# Filter proximity data for specific ports
proximity_filtered = get_trial_proximities(animal_id, session, ports=[1, 2, 3])
print(f"Proximity data for ports [1, 2, 3]: {len(proximity_filtered)} events")
print(f"Events per port: {proximity_filtered['port'].value_counts()}")

## 6. Analysis and Performance Functions

These functions compute performance metrics and provide session analysis.

In [ ]:
# get_performance() - Calculate performance as ratio of reward to total decisive trials
# Parameters: animal_id (int), session (int), trials (List[int])
# Returns: Performance ratio (0-1) or None
performance = get_performance(animal_id, session)
print(f"Performance for animal {animal_id}, session {session}: {performance:.3f}")

# Calculate performance for specific trials
trial_subset = list(range(1, 101))  # First 100 trials
performance_subset = get_performance(animal_id, session, trials=trial_subset)
print(f"Performance for first 100 trials: {performance_subset:.3f}")

In [ ]:
# trials_per_session() - Returns the number of trials per session
# Parameters: animal_id (int), min_trials (int), format (str)
# Returns: DataFrame with trials_count column
session_trial_counts = trials_per_session(animal_id, min_trials=10)
print(f"Trial counts per session for animal {animal_id}:")
print(session_trial_counts.head())
print("\nSummary statistics:")
print(session_trial_counts['trials_count'].describe())

In [ ]:
# session_summary() - Print comprehensive summary of a session
# Parameters: animal_id (int), session (int)
# Returns: None (prints summary)
print(f"Comprehensive session summary for animal {animal_id}, session {session}:")
print("=" * 60)
session_summary(animal_id, session)

## 7. Complete Data Loading Example

This example shows how to load all available data for a comprehensive analysis.

In [ ]:
# Load all available data for a session
print(f"Loading complete dataset for animal {animal_id}, session {session}...")
print("=" * 60)

# Session-level data
sessions_data = get_sessions(animal_id)
session_info = get_session_classes(animal_id, session)
session_dur = get_session_duration(animal_id, session)
task_file = get_session_task(animal_id, session, save_file=False)
session_versions = get_session_version(animal_id, session)

# Trial-level data
trials_data = get_trials(animal_id, session)
states_data = get_trial_states(animal_id, session)
experiment_data = get_trial_experiment(animal_id, session)
behavior_data = get_trial_behavior(animal_id, session)
stimulus_data = get_trial_stimulus(animal_id, session)

# Behavioral events
licks_data = get_trial_licks(animal_id, session)
proximities_data = get_trial_proximities(animal_id, session)

# Performance metrics
performance_score = get_performance(animal_id, session)

print("Data loading complete!")
print(f"Sessions available: {len(sessions_data)}")
print(f"Trials in session {session}: {len(trials_data)}")
print(f"States: {len(states_data)}")
print(f"Lick events: {len(licks_data)}")
print(f"Proximity events: {len(proximities_data)}")
print(f"Session performance: {performance_score:.3f}")
print(f"Session duration: {session_dur}")
print(f"Task file: {task_file}")
print(f"Code version records: {len(session_versions)}")

## 8. Data Format Options

Most functions support both DataFrame and DataJoint formats for flexibility.

In [ ]:
# Compare DataFrame vs DataJoint formats
print("Data format comparison:")
print("=" * 40)

# DataFrame format (default)
trials_df = get_trials(animal_id, session, format="df")
print(f"DataFrame format: {type(trials_df)}")
print(f"Shape: {trials_df.shape}")
print(f"Columns: {list(trials_df.columns)}")

# DataJoint format
trials_dj = get_trials(animal_id, session, format="dj")
print(f"\nDataJoint format: {type(trials_dj)}")
print(f"Length: {len(trials_dj)}")

# Convert DataJoint to DataFrame if needed
trials_from_dj = trials_dj.fetch(format="frame").reset_index()
print(f"\nConverted back to DataFrame: {type(trials_from_dj)}")
print(f"Shape: {trials_from_dj.shape}")

## 9. Error Handling and Edge Cases

Examples of how functions handle common edge cases.

In [ ]:
# Handle cases with no data
print("Testing edge cases:")
print("=" * 30)

# Try to get performance for a session with no decisive trials
# (This would return None if no reward/punish trials exist)
try:
    perf_result = get_performance(animal_id, session, trials=[999999])  # Non-existent trial
    print(f"Performance for non-existent trial: {perf_result}")
except Exception as e:
    print(f"Expected error for non-existent trial: {type(e).__name__}")

# Try to get data for future date range
future_sessions = get_sessions(animal_id, from_date="2030-01-01", to_date="2030-12-31")
print(f"Sessions in future date range: {len(future_sessions)}")

## Summary

This notebook demonstrated all the data loading functions available in the ethopy_analysis package:

### Configuration & Setup
- `load_config()` - Load configuration settings
- `get_database_config()` - Get database configuration
- `get_setup()` - Get animal/session from setup ID

### Database Schemas
- `get_schema()` - Get individual schema
- `get_all_schemas()` - Get all schemas at once

### Session Data
- `get_sessions()` - Get sessions with filtering
- `get_session_classes()` - Get session metadata
- `get_session_duration()` - Get session duration
- `get_session_task()` - Get task file info
- `get_session_version()` - Get code version (git hash / package version) records

### Trial Data
- `get_trials()` - Get trial data
- `get_trial_states()` - Get state onset times
- `get_trial_experiment()` - Get experiment conditions
- `get_trial_behavior()` - Get behavior conditions
- `get_trial_stimulus()` - Get stimulus conditions

### Behavioral Events
- `get_trial_licks()` - Get lick events
- `get_trial_proximities()` - Get proximity events

### Analysis Functions
- `get_performance()` - Calculate performance metrics
- `trials_per_session()` - Count trials per session
- `session_summary()` - Print session summary

All functions support flexible data formats (DataFrame/DataJoint) and include proper error handling for edge cases.